<a href="https://colab.research.google.com/github/AshokGit544/Enterprise-Utility-Lakehouse-Pipeline1/blob/main/Copy_of_Enterprise_Utility_Lakehouse_Pipeline1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -y
!apt-get install -y default-jdk wget
!pip install -q pyspark pandas numpy faker matplotlib scikit-learn pyarrow

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,628 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,933 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,615 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,870 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [6,994 kB]
Get:14 ht

In [ ]:
!java -version

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


In [ ]:
import os
import subprocess

java_path = subprocess.check_output("readlink -f $(which java) | sed 's:/bin/java::'", shell=True).decode().strip()
os.environ["JAVA_HOME"] = java_path

print("JAVA_HOME =", os.environ["JAVA_HOME"])

JAVA_HOME = /usr/lib/jvm/java-17-openjdk-amd64


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EnterpriseUtilityLakehousePipeline")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print(spark.version)

3.5.1


In [ ]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
os.environ["PATH"] += ":/content/spark-3.5.1-bin-hadoop3/bin"

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EnterpriseUtilityLakehousePipeline")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

In [ ]:
import random
import json
import math
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from faker import Faker
import matplotlib.pyplot as plt

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

fake = Faker()
random.seed(42)
np.random.seed(42)

BASE_PATH = Path("/content/enterprise_utility_lakehouse_project")
RAW_PATH = BASE_PATH / "data/raw"
BRONZE_PATH = BASE_PATH / "data/bronze"
SILVER_PATH = BASE_PATH / "data/silver"
GOLD_PATH = BASE_PATH / "data/gold"
OUTPUT_PATH = BASE_PATH / "outputs"

for p in [RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, OUTPUT_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print("Project folders created at:", BASE_PATH)

Project folders created at: /content/enterprise_utility_lakehouse_project


In [ ]:
num_customers = 5000
num_assets = 1200
num_outages = 8000
num_weather = 3650
num_billing = 24000

regions = ["North", "South", "East", "West", "Central"]
outage_causes = ["Storm", "Equipment Failure", "Vegetation", "Animal Contact", "Unknown"]

from datetime import datetime, timedelta
import random
import numpy as np
import pandas as pd

start_date = datetime(2023, 1, 1)

# Outage events
outages = []

for i in range(1, num_outages + 1):
    cust = random.randint(1, num_customers)
    asset = random.randint(1, num_assets)
    outage_date = start_date + timedelta(days=random.randint(0, 364))
    duration = max(5, int(np.random.normal(90, 40)))

    outages.append({
        "outage_id": f"OUT{i:07d}",
        "customer_id": f"CUST{cust:06d}",
        "asset_id": f"AST{asset:05d}",
        "outage_date": outage_date.strftime("%Y-%m-%d"),
        "duration_minutes": duration,
        "cause": random.choice(outage_causes),
        "restoration_status": random.choice(["Restored", "Restored", "Pending"])
    })

outages_df = pd.DataFrame(outages)
outages_df.to_csv(RAW_PATH / "outages.csv", index=False)

# Weather
weather = []

for i in range(num_weather):
    weather_date = start_date + timedelta(days=i % 365)

    weather.append({
        "weather_date": weather_date.strftime("%Y-%m-%d"),
        "region": random.choice(regions),
        "temperature_f": round(float(np.random.normal(62, 18)), 2),
        "rainfall_mm": round(float(max(0, np.random.normal(3, 5))), 2),
        "wind_speed_mph": round(float(max(0, np.random.normal(12, 6))), 2),
        "storm_flag": random.choice([0, 0, 0, 1])
    })

weather_df = pd.DataFrame(weather)
weather_df.to_csv(RAW_PATH / "weather.csv", index=False)

# Billing / SAP FICO-like finance data
billing = []

for i in range(1, num_billing + 1):
    cust = random.randint(1, num_customers)
    bill_date = start_date + timedelta(days=random.randint(0, 364))
    amount = max(40, np.random.normal(145, 55))

    billing.append({
        "billing_id": f"BILL{i:07d}",
        "customer_id": f"CUST{cust:06d}",
        "sap_fico_doc_id": f"SAPFICO{i:08d}",
        "bill_date": bill_date.strftime("%Y-%m-%d"),
        "bill_amount": round(float(amount), 2),
        "payment_status": random.choice(["Paid", "Paid", "Paid", "Pending", "Overdue"]),
        "gl_account": random.choice(["400100", "400200", "400300", "410100"])
    })

billing_df = pd.DataFrame(billing)
billing_df.to_csv(RAW_PATH / "billing.csv", index=False)

print("Raw datasets generated successfully.")
for file in RAW_PATH.glob("*.csv"):
    print(file.name)

Raw datasets generated successfully.
weather.csv
outages.csv
billing.csv


In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import random
import numpy as np
import pandas as pd
import os

BASE_PATH = Path("/content/enterprise_utility_lakehouse_project")
RAW_PATH = BASE_PATH / "data" / "raw"
BRONZE_PATH = BASE_PATH / "data" / "bronze"
SILVER_PATH = BASE_PATH / "data" / "silver"
GOLD_PATH = BASE_PATH / "data" / "gold"
OUTPUT_PATH = BASE_PATH / "outputs"

for p in [RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, OUTPUT_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH:", RAW_PATH)
print("Files currently in raw path:", list(RAW_PATH.glob("*")))

BASE_PATH: /content/enterprise_utility_lakehouse_project
RAW_PATH: /content/enterprise_utility_lakehouse_project/data/raw
Files currently in raw path: [PosixPath('/content/enterprise_utility_lakehouse_project/data/raw/weather.csv'), PosixPath('/content/enterprise_utility_lakehouse_project/data/raw/outages.csv'), PosixPath('/content/enterprise_utility_lakehouse_project/data/raw/billing.csv')]


In [ ]:
import random
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

num_customers = 5000
num_assets = 1200
num_meter_records = 100000
num_outages = 8000
num_weather = 3650
num_billing = 24000

regions = ["North", "South", "East", "West", "Central"]
customer_types = ["Residential", "Commercial", "Industrial"]
asset_types = ["Transformer", "Feeder", "Substation", "Meter", "Breaker"]
outage_causes = ["Storm", "Equipment Failure", "Vegetation", "Animal Contact", "Unknown"]

start_date = datetime(2023, 1, 1)

# -------------------------
# Customers
# -------------------------
customers = []
for i in range(1, num_customers + 1):
    customers.append({
        "customer_id": f"CUST{i:06d}",
        "customer_name": f"Customer_{i}",
        "customer_type": random.choice(customer_types),
        "region": random.choice(regions),
        "account_start_date": (start_date - timedelta(days=random.randint(100, 3000))).strftime("%Y-%m-%d"),
        "status": random.choice(["Active", "Active", "Active", "Inactive"])
    })

customers_df = pd.DataFrame(customers)
customers_df.to_csv(RAW_PATH / "customers.csv", index=False)

# -------------------------
# Assets
# -------------------------
assets = []
for i in range(1, num_assets + 1):
    assets.append({
        "asset_id": f"AST{i:05d}",
        "asset_type": random.choice(asset_types),
        "region": random.choice(regions),
        "install_date": (start_date - timedelta(days=random.randint(365, 5000))).strftime("%Y-%m-%d"),
        "asset_status": random.choice(["Active", "Active", "Maintenance", "Inactive"])
    })

assets_df = pd.DataFrame(assets)
assets_df.to_csv(RAW_PATH / "assets.csv", index=False)

# -------------------------
# Meter usage
# -------------------------
meter_records = []
for _ in range(num_meter_records):
    cust = random.randint(1, num_customers)
    reading_date = start_date + timedelta(days=random.randint(0, 364))
    usage = max(5, np.random.normal(35, 12))

    meter_records.append({
        "customer_id": f"CUST{cust:06d}",
        "meter_id": f"MTR{cust:06d}",
        "reading_date": reading_date.strftime("%Y-%m-%d"),
        "kwh_usage": round(float(usage), 2),
        "voltage_flag": random.choice(["Normal", "Normal", "High", "Low"]),
        "estimated_read_flag": random.choice([0, 0, 0, 1])
    })

meter_df = pd.DataFrame(meter_records)
meter_df.loc[10, "kwh_usage"] = None
meter_df = pd.concat([meter_df, meter_df.iloc[:50]], ignore_index=True)
meter_df.to_csv(RAW_PATH / "meter_usage.csv", index=False)

# -------------------------
# Outages
# -------------------------
outages = []
for i in range(1, num_outages + 1):
    cust = random.randint(1, num_customers)
    asset = random.randint(1, num_assets)
    outage_date = start_date + timedelta(days=random.randint(0, 364))
    duration = max(5, int(np.random.normal(90, 40)))

    outages.append({
        "outage_id": f"OUT{i:07d}",
        "customer_id": f"CUST{cust:06d}",
        "asset_id": f"AST{asset:05d}",
        "outage_date": outage_date.strftime("%Y-%m-%d"),
        "duration_minutes": duration,
        "cause": random.choice(outage_causes),
        "restoration_status": random.choice(["Restored", "Restored", "Pending"])
    })

outages_df = pd.DataFrame(outages)
outages_df.to_csv(RAW_PATH / "outages.csv", index=False)

# -------------------------
# Weather
# -------------------------
weather = []
for i in range(num_weather):
    weather_date = start_date + timedelta(days=i % 365)

    weather.append({
        "weather_date": weather_date.strftime("%Y-%m-%d"),
        "region": random.choice(regions),
        "temperature_f": round(float(np.random.normal(62, 18)), 2),
        "rainfall_mm": round(float(max(0, np.random.normal(3, 5))), 2),
        "wind_speed_mph": round(float(max(0, np.random.normal(12, 6))), 2),
        "storm_flag": random.choice([0, 0, 0, 1])
    })

weather_df = pd.DataFrame(weather)
weather_df.to_csv(RAW_PATH / "weather.csv", index=False)

# -------------------------
# Billing
# -------------------------
billing = []
for i in range(1, num_billing + 1):
    cust = random.randint(1, num_customers)
    bill_date = start_date + timedelta(days=random.randint(0, 364))
    amount = max(40, np.random.normal(145, 55))

    billing.append({
        "billing_id": f"BILL{i:07d}",
        "customer_id": f"CUST{cust:06d}",
        "sap_fico_doc_id": f"SAPFICO{i:08d}",
        "bill_date": bill_date.strftime("%Y-%m-%d"),
        "bill_amount": round(float(amount), 2),
        "payment_status": random.choice(["Paid", "Paid", "Paid", "Pending", "Overdue"]),
        "gl_account": random.choice(["400100", "400200", "400300", "410100"])
    })

billing_df = pd.DataFrame(billing)
billing_df.to_csv(RAW_PATH / "billing.csv", index=False)

print("Raw datasets generated successfully.")
for file in RAW_PATH.glob("*.csv"):
    print(file.name, "-", file.stat().st_size, "bytes")

Raw datasets generated successfully.
assets.csv - 52058 bytes
weather.csv - 128403 bytes
outages.csv - 525227 bytes
meter_usage.csv - 4564704 bytes
billing.csv - 1677002 bytes
customers.csv - 303170 bytes


In [ ]:
required_files = [
    "customers.csv",
    "assets.csv",
    "meter_usage.csv",
    "outages.csv",
    "weather.csv",
    "billing.csv"
]

missing_files = [f for f in required_files if not (RAW_PATH / f).exists()]

if missing_files:
    print("Missing files:", missing_files)
else:
    print("All raw files are present.")
    for f in required_files:
        print(RAW_PATH / f)

All raw files are present.
/content/enterprise_utility_lakehouse_project/data/raw/customers.csv
/content/enterprise_utility_lakehouse_project/data/raw/assets.csv
/content/enterprise_utility_lakehouse_project/data/raw/meter_usage.csv
/content/enterprise_utility_lakehouse_project/data/raw/outages.csv
/content/enterprise_utility_lakehouse_project/data/raw/weather.csv
/content/enterprise_utility_lakehouse_project/data/raw/billing.csv


In [ ]:
customers_bronze = spark.read.option("header", "true").option("inferSchema", "true").csv(str(RAW_PATH / "customers.csv"))
assets_bronze = spark.read.option("header", "true").option("inferSchema", "true").csv(str(RAW_PATH / "assets.csv"))
meter_bronze = spark.read.option("header", "true").option("inferSchema", "true").csv(str(RAW_PATH / "meter_usage.csv"))
outages_bronze = spark.read.option("header", "true").option("inferSchema", "true").csv(str(RAW_PATH / "outages.csv"))
weather_bronze = spark.read.option("header", "true").option("inferSchema", "true").csv(str(RAW_PATH / "weather.csv"))
billing_bronze = spark.read.option("header", "true").option("inferSchema", "true").csv(str(RAW_PATH / "billing.csv"))

print("Bronze layer row counts")
print("customers:", customers_bronze.count())
print("assets:", assets_bronze.count())
print("meter:", meter_bronze.count())
print("outages:", outages_bronze.count())
print("weather:", weather_bronze.count())
print("billing:", billing_bronze.count())

Bronze layer row counts
customers: 5000
assets: 1200
meter: 100050
outages: 8000
weather: 3650
billing: 24000


In [ ]:
customers_bronze.show(5, truncate=False)
assets_bronze.show(5, truncate=False)
meter_bronze.show(5, truncate=False)
outages_bronze.show(5, truncate=False)
weather_bronze.show(5, truncate=False)
billing_bronze.show(5, truncate=False)

+-----------+-------------+-------------+-------+------------------+--------+
|customer_id|customer_name|customer_type|region |account_start_date|status  |
+-----------+-------------+-------------+-------+------------------+--------+
|CUST000001 |Customer_1   |Industrial   |North  |2022-06-13        |Active  |
|CUST000002 |Customer_2   |Residential  |South  |2021-03-01        |Active  |
|CUST000003 |Customer_3   |Industrial   |Central|2021-10-02        |Inactive|
|CUST000004 |Customer_4   |Residential  |North  |2021-09-05        |Active  |
|CUST000005 |Customer_5   |Residential  |Central|2015-12-24        |Active  |
+-----------+-------------+-------------+-------+------------------+--------+
only showing top 5 rows

+--------+-----------+-------+------------+------------+
|asset_id|asset_type |region |install_date|asset_status|
+--------+-----------+-------+------------+------------+
|AST00001|Feeder     |East   |2011-04-01  |Active      |
|AST00002|Feeder     |East   |2011-08-25  |In

In [ ]:
customers_bronze.write.mode("overwrite").parquet(str(BRONZE_PATH / "customers"))
assets_bronze.write.mode("overwrite").parquet(str(BRONZE_PATH / "assets"))
meter_bronze.write.mode("overwrite").parquet(str(BRONZE_PATH / "meter_usage"))
outages_bronze.write.mode("overwrite").parquet(str(BRONZE_PATH / "outages"))
weather_bronze.write.mode("overwrite").parquet(str(BRONZE_PATH / "weather"))
billing_bronze.write.mode("overwrite").parquet(str(BRONZE_PATH / "billing"))

print("Bronze layer saved successfully.")

Bronze layer saved successfully.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
def print_null_counts(df, df_name):
    exprs = [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]
    print(f"\nNull counts for {df_name}")
    df.select(exprs).show(truncate=False)


def drop_duplicate_records(df, subset_cols, df_name="df"):
    before = df.count()
    deduped = df.dropDuplicates(subset_cols)
    after = deduped.count()
    print(f"{df_name} duplicates removed: {before - after}")
    return deduped

In [ ]:
customers_silver = (
    customers_bronze
    .withColumn("account_start_date", F.to_date("account_start_date"))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("customer_type", F.trim(F.col("customer_type")))
    .withColumn("region", F.trim(F.col("region")))
    .withColumn("status", F.trim(F.col("status")))
    .filter(F.col("customer_id").isNotNull())
)

customers_silver = drop_duplicate_records(customers_silver, ["customer_id"], "customers_silver")
print_null_counts(customers_silver, "customers_silver")
customers_silver.show(5, truncate=False)

customers_silver duplicates removed: 0

Null counts for customers_silver
+-----------+-------------+-------------+------+------------------+------+
|customer_id|customer_name|customer_type|region|account_start_date|status|
+-----------+-------------+-------------+------+------------------+------+
|0          |0            |0            |0     |0                 |0     |
+-----------+-------------+-------------+------+------------------+------+

+-----------+-------------+-------------+-------+------------------+--------+
|customer_id|customer_name|customer_type|region |account_start_date|status  |
+-----------+-------------+-------------+-------+------------------+--------+
|CUST000001 |Customer_1   |Industrial   |North  |2022-06-13        |Active  |
|CUST000002 |Customer_2   |Residential  |South  |2021-03-01        |Active  |
|CUST000003 |Customer_3   |Industrial   |Central|2021-10-02        |Inactive|
|CUST000004 |Customer_4   |Residential  |North  |2021-09-05        |Active  |
|CUST

In [ ]:
assets_silver = (
    assets_bronze
    .withColumn("install_date", F.to_date("install_date"))
    .withColumn("asset_id", F.trim(F.col("asset_id")))
    .withColumn("asset_type", F.trim(F.col("asset_type")))
    .withColumn("region", F.trim(F.col("region")))
    .withColumn("asset_status", F.trim(F.col("asset_status")))
    .filter(F.col("asset_id").isNotNull())
)

assets_silver = drop_duplicate_records(assets_silver, ["asset_id"], "assets_silver")
print_null_counts(assets_silver, "assets_silver")
assets_silver.show(5, truncate=False)

assets_silver duplicates removed: 0

Null counts for assets_silver
+--------+----------+------+------------+------------+
|asset_id|asset_type|region|install_date|asset_status|
+--------+----------+------+------------+------------+
|0       |0         |0     |0           |0           |
+--------+----------+------+------------+------------+

+--------+-----------+-------+------------+------------+
|asset_id|asset_type |region |install_date|asset_status|
+--------+-----------+-------+------------+------------+
|AST00001|Feeder     |East   |2011-04-01  |Active      |
|AST00002|Feeder     |East   |2011-08-25  |Inactive    |
|AST00003|Transformer|North  |2021-01-05  |Active      |
|AST00004|Transformer|Central|2019-04-03  |Active      |
|AST00005|Feeder     |North  |2012-10-01  |Active      |
+--------+-----------+-------+------------+------------+
only showing top 5 rows



In [ ]:
meter_silver = (
    meter_bronze
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("meter_id", F.trim(F.col("meter_id")))
    .withColumn("reading_date", F.to_date("reading_date"))
    .withColumn("kwh_usage", F.col("kwh_usage").cast("double"))
    .withColumn("voltage_flag", F.trim(F.col("voltage_flag")))
    .withColumn("estimated_read_flag", F.col("estimated_read_flag").cast("int"))
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("meter_id").isNotNull())
    .filter(F.col("reading_date").isNotNull())
    .fillna({"kwh_usage": 0.0, "estimated_read_flag": 0})
)

meter_silver = meter_silver.filter(F.col("kwh_usage") >= 0)
meter_silver = drop_duplicate_records(
    meter_silver,
    ["customer_id", "meter_id", "reading_date"],
    "meter_silver"
)

print_null_counts(meter_silver, "meter_silver")
meter_silver.show(5, truncate=False)

meter_silver duplicates removed: 2689

Null counts for meter_silver
+-----------+--------+------------+---------+------------+-------------------+
|customer_id|meter_id|reading_date|kwh_usage|voltage_flag|estimated_read_flag|
+-----------+--------+------------+---------+------------+-------------------+
|0          |0       |0           |0        |0           |0                  |
+-----------+--------+------------+---------+------------+-------------------+

+-----------+---------+------------+---------+------------+-------------------+
|customer_id|meter_id |reading_date|kwh_usage|voltage_flag|estimated_read_flag|
+-----------+---------+------------+---------+------------+-------------------+
|CUST000001 |MTR000001|2023-01-01  |28.97    |Normal      |0                  |
|CUST000001 |MTR000001|2023-02-06  |30.36    |High        |0                  |
|CUST000001 |MTR000001|2023-06-14  |25.15    |Normal      |0                  |
|CUST000001 |MTR000001|2023-06-25  |43.84    |Low       

In [ ]:
outages_silver = (
    outages_bronze
    .withColumn("outage_id", F.trim(F.col("outage_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("asset_id", F.trim(F.col("asset_id")))
    .withColumn("outage_date", F.to_date("outage_date"))
    .withColumn("duration_minutes", F.col("duration_minutes").cast("int"))
    .withColumn("cause", F.trim(F.col("cause")))
    .withColumn("restoration_status", F.trim(F.col("restoration_status")))
    .filter(F.col("outage_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("asset_id").isNotNull())
    .filter(F.col("outage_date").isNotNull())
    .fillna({"duration_minutes": 0})
)

outages_silver = outages_silver.filter(F.col("duration_minutes") >= 0)
outages_silver = drop_duplicate_records(outages_silver, ["outage_id"], "outages_silver")

print_null_counts(outages_silver, "outages_silver")
outages_silver.show(5, truncate=False)

outages_silver duplicates removed: 0

Null counts for outages_silver
+---------+-----------+--------+-----------+----------------+-----+------------------+
|outage_id|customer_id|asset_id|outage_date|duration_minutes|cause|restoration_status|
+---------+-----------+--------+-----------+----------------+-----+------------------+
|0        |0          |0       |0          |0               |0    |0                 |
+---------+-----------+--------+-----------+----------------+-----+------------------+

+----------+-----------+--------+-----------+----------------+-----------------+------------------+
|outage_id |customer_id|asset_id|outage_date|duration_minutes|cause            |restoration_status|
+----------+-----------+--------+-----------+----------------+-----------------+------------------+
|OUT0000001|CUST001943 |AST00913|2023-04-22 |131             |Unknown          |Restored          |
|OUT0000002|CUST004954 |AST00036|2023-08-09 |43              |Equipment Failure|Pending        

In [ ]:
weather_silver = (
    weather_bronze
    .withColumn("weather_date", F.to_date("weather_date"))
    .withColumn("region", F.trim(F.col("region")))
    .withColumn("temperature_f", F.col("temperature_f").cast("double"))
    .withColumn("rainfall_mm", F.col("rainfall_mm").cast("double"))
    .withColumn("wind_speed_mph", F.col("wind_speed_mph").cast("double"))
    .withColumn("storm_flag", F.col("storm_flag").cast("int"))
    .filter(F.col("weather_date").isNotNull())
    .filter(F.col("region").isNotNull())
    .fillna({
        "temperature_f": 0.0,
        "rainfall_mm": 0.0,
        "wind_speed_mph": 0.0,
        "storm_flag": 0
    })
)

weather_silver = drop_duplicate_records(weather_silver, ["weather_date", "region"], "weather_silver")
print_null_counts(weather_silver, "weather_silver")
weather_silver.show(5, truncate=False)

weather_silver duplicates removed: 2018

Null counts for weather_silver
+------------+------+-------------+-----------+--------------+----------+
|weather_date|region|temperature_f|rainfall_mm|wind_speed_mph|storm_flag|
+------------+------+-------------+-----------+--------------+----------+
|0           |0     |0            |0          |0             |0         |
+------------+------+-------------+-----------+--------------+----------+

+------------+-------+-------------+-----------+--------------+----------+
|weather_date|region |temperature_f|rainfall_mm|wind_speed_mph|storm_flag|
+------------+-------+-------------+-----------+--------------+----------+
|2023-01-08  |West   |85.53        |0.0        |5.91          |1         |
|2023-01-16  |South  |47.62        |11.16      |18.2          |0         |
|2023-01-28  |East   |38.9         |0.0        |20.6          |0         |
|2023-02-03  |Central|35.72        |3.59       |12.27         |1         |
|2023-02-09  |North  |72.83     

In [ ]:
billing_silver = (
    billing_bronze
    .withColumn("billing_id", F.trim(F.col("billing_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("sap_fico_doc_id", F.trim(F.col("sap_fico_doc_id")))
    .withColumn("bill_date", F.to_date("bill_date"))
    .withColumn("bill_amount", F.col("bill_amount").cast("double"))
    .withColumn("payment_status", F.trim(F.col("payment_status")))
    .withColumn("gl_account", F.trim(F.col("gl_account")))
    .filter(F.col("billing_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("bill_date").isNotNull())
    .fillna({"bill_amount": 0.0})
)

billing_silver = billing_silver.filter(F.col("bill_amount") >= 0)
billing_silver = drop_duplicate_records(billing_silver, ["billing_id"], "billing_silver")

print_null_counts(billing_silver, "billing_silver")
billing_silver.show(5, truncate=False)

billing_silver duplicates removed: 0

Null counts for billing_silver
+----------+-----------+---------------+---------+-----------+--------------+----------+
|billing_id|customer_id|sap_fico_doc_id|bill_date|bill_amount|payment_status|gl_account|
+----------+-----------+---------------+---------+-----------+--------------+----------+
|0         |0          |0              |0        |0          |0             |0         |
+----------+-----------+---------------+---------+-----------+--------------+----------+

+-----------+-----------+---------------+----------+-----------+--------------+----------+
|billing_id |customer_id|sap_fico_doc_id|bill_date |bill_amount|payment_status|gl_account|
+-----------+-----------+---------------+----------+-----------+--------------+----------+
|BILL0000001|CUST004396 |SAPFICO00000001|2023-05-14|197.49     |Paid          |400100    |
|BILL0000002|CUST004492 |SAPFICO00000002|2023-06-16|184.41     |Overdue       |400200    |
|BILL0000003|CUST004213 |SAPFI

In [ ]:
customers_silver.write.mode("overwrite").parquet(str(SILVER_PATH / "customers"))
assets_silver.write.mode("overwrite").parquet(str(SILVER_PATH / "assets"))
meter_silver.write.mode("overwrite").parquet(str(SILVER_PATH / "meter_usage"))
outages_silver.write.mode("overwrite").parquet(str(SILVER_PATH / "outages"))
weather_silver.write.mode("overwrite").parquet(str(SILVER_PATH / "weather"))
billing_silver.write.mode("overwrite").parquet(str(SILVER_PATH / "billing"))

print("Silver layer saved successfully.")

Silver layer saved successfully.


In [ ]:
print("Silver layer row counts")
print("customers:", customers_silver.count())
print("assets:", assets_silver.count())
print("meter:", meter_silver.count())
print("outages:", outages_silver.count())
print("weather:", weather_silver.count())
print("billing:", billing_silver.count())

Silver layer row counts
customers: 5000
assets: 1200
meter: 97361
outages: 8000
weather: 1632
billing: 24000


In [ ]:
meter_daily = (
    meter_silver
    .groupBy("customer_id", "reading_date")
    .agg(
        F.sum("kwh_usage").alias("daily_kwh_usage"),
        F.max("estimated_read_flag").alias("has_estimated_read")
    )
)
meter_daily.show(5, truncate=False)

+-----------+------------+---------------+------------------+
|customer_id|reading_date|daily_kwh_usage|has_estimated_read|
+-----------+------------+---------------+------------------+
|CUST002013 |2023-05-21  |38.55          |0                 |
|CUST000048 |2023-12-22  |34.11          |1                 |
|CUST002260 |2023-04-19  |26.84          |0                 |
|CUST003276 |2023-01-13  |44.93          |0                 |
|CUST004276 |2023-12-15  |15.65          |0                 |
+-----------+------------+---------------+------------------+
only showing top 5 rows



In [ ]:
outage_daily = (
    outages_silver
    .groupBy("customer_id", "outage_date")
    .agg(
        F.count("outage_id").alias("daily_outage_count"),
        F.sum("duration_minutes").alias("daily_outage_minutes")
    )
    .withColumnRenamed("outage_date", "event_date")
)
outage_daily.show(5, truncate=False)

+-----------+----------+------------------+--------------------+
|customer_id|event_date|daily_outage_count|daily_outage_minutes|
+-----------+----------+------------------+--------------------+
|CUST004810 |2023-09-01|1                 |76                  |
|CUST002809 |2023-05-06|1                 |22                  |
|CUST004424 |2023-04-28|1                 |135                 |
|CUST003431 |2023-09-16|1                 |61                  |
|CUST003463 |2023-11-02|1                 |87                  |
+-----------+----------+------------------+--------------------+
only showing top 5 rows



In [ ]:
billing_daily = (
    billing_silver
    .groupBy("customer_id", "bill_date")
    .agg(
        F.sum("bill_amount").alias("daily_billed_amount"),
        F.count("billing_id").alias("daily_bill_count")
    )
    .withColumnRenamed("bill_date", "event_date")
)
billing_daily.show(5, truncate=False)

+-----------+----------+-------------------+----------------+
|customer_id|event_date|daily_billed_amount|daily_bill_count|
+-----------+----------+-------------------+----------------+
|CUST004200 |2023-01-11|183.75             |1               |
|CUST002437 |2023-07-12|81.75              |1               |
|CUST002508 |2023-06-02|178.91             |1               |
|CUST004431 |2023-12-30|147.17             |1               |
|CUST002225 |2023-11-29|199.25             |1               |
+-----------+----------+-------------------+----------------+
only showing top 5 rows



In [ ]:
meter_with_customer = (
    meter_daily.alias("m")
    .join(
        customers_silver.select("customer_id", "region", "customer_type", "status").alias("c"),
        on="customer_id",
        how="left"
    )
    .withColumnRenamed("reading_date", "event_date")
)

meter_with_customer.show(5, truncate=False)

+-----------+----------+---------------+------------------+-------+-------------+------+
|customer_id|event_date|daily_kwh_usage|has_estimated_read|region |customer_type|status|
+-----------+----------+---------------+------------------+-------+-------------+------+
|CUST002013 |2023-05-21|38.55          |0                 |West   |Commercial   |Active|
|CUST000048 |2023-12-22|34.11          |1                 |East   |Industrial   |Active|
|CUST002260 |2023-04-19|26.84          |0                 |Central|Industrial   |Active|
|CUST003276 |2023-01-13|44.93          |0                 |East   |Commercial   |Active|
|CUST004276 |2023-12-15|15.65          |0                 |Central|Residential  |Active|
+-----------+----------+---------------+------------------+-------+-------------+------+
only showing top 5 rows



In [ ]:
weather_daily = weather_silver.select(
    F.col("weather_date").alias("event_date"),
    "region",
    "temperature_f",
    "rainfall_mm",
    "wind_speed_mph",
    "storm_flag"
)

weather_daily.show(5, truncate=False)

+----------+-------+-------------+-----------+--------------+----------+
|event_date|region |temperature_f|rainfall_mm|wind_speed_mph|storm_flag|
+----------+-------+-------------+-----------+--------------+----------+
|2023-01-08|West   |85.53        |0.0        |5.91          |1         |
|2023-01-16|South  |47.62        |11.16      |18.2          |0         |
|2023-01-28|East   |38.9         |0.0        |20.6          |0         |
|2023-02-03|Central|35.72        |3.59       |12.27         |1         |
|2023-02-09|North  |72.83        |1.06       |12.6          |0         |
+----------+-------+-------------+-----------+--------------+----------+
only showing top 5 rows



In [ ]:
enterprise_gold = (
    meter_with_customer.alias("m")
    .join(
        outage_daily.alias("o"),
        on=["customer_id", "event_date"],
        how="left"
    )
    .join(
        billing_daily.alias("b"),
        on=["customer_id", "event_date"],
        how="left"
    )
    .join(
        weather_daily.alias("w"),
        on=["event_date", "region"],
        how="left"
    )
    .fillna({
        "daily_outage_count": 0,
        "daily_outage_minutes": 0,
        "daily_billed_amount": 0.0,
        "daily_bill_count": 0,
        "temperature_f": 0.0,
        "rainfall_mm": 0.0,
        "wind_speed_mph": 0.0,
        "storm_flag": 0
    })
)

enterprise_gold.show(10, truncate=False)
print("Gold rows:", enterprise_gold.count())

+----------+-------+-----------+---------------+------------------+-------------+--------+------------------+--------------------+-------------------+----------------+-------------+-----------+--------------+----------+
|event_date|region |customer_id|daily_kwh_usage|has_estimated_read|customer_type|status  |daily_outage_count|daily_outage_minutes|daily_billed_amount|daily_bill_count|temperature_f|rainfall_mm|wind_speed_mph|storm_flag|
+----------+-------+-----------+---------------+------------------+-------------+--------+------------------+--------------------+-------------------+----------------+-------------+-----------+--------------+----------+
|2023-05-21|West   |CUST002013 |38.55          |0                 |Commercial   |Active  |0                 |0                   |0.0                |0               |46.01        |0.0        |7.18          |0         |
|2023-12-22|East   |CUST000048 |34.11          |1                 |Industrial   |Active  |0                 |0          

In [ ]:
window_spec = Window.partitionBy("customer_id").orderBy("event_date")

enterprise_gold_features = (
    enterprise_gold
    .withColumn("usage_7day_avg", F.avg("daily_kwh_usage").over(window_spec.rowsBetween(-6, 0)))
    .withColumn("usage_30day_avg", F.avg("daily_kwh_usage").over(window_spec.rowsBetween(-29, 0)))
    .withColumn("prev_day_usage", F.lag("daily_kwh_usage", 1).over(window_spec))
    .withColumn("is_high_outage_day", F.when(F.col("daily_outage_minutes") > 120, 1).otherwise(0))
    .withColumn("month", F.month("event_date"))
    .withColumn("day_of_week", F.dayofweek("event_date"))
)

enterprise_gold_features.show(10, truncate=False)

+----------+------+-----------+---------------+------------------+-------------+------+------------------+--------------------+-------------------+----------------+-------------+-----------+--------------+----------+------------------+------------------+--------------+------------------+-----+-----------+
|event_date|region|customer_id|daily_kwh_usage|has_estimated_read|customer_type|status|daily_outage_count|daily_outage_minutes|daily_billed_amount|daily_bill_count|temperature_f|rainfall_mm|wind_speed_mph|storm_flag|usage_7day_avg    |usage_30day_avg   |prev_day_usage|is_high_outage_day|month|day_of_week|
+----------+------+-----------+---------------+------------------+-------------+------+------------------+--------------------+-------------------+----------------+-------------+-----------+--------------+----------+------------------+------------------+--------------+------------------+-----+-----------+
|2023-01-04|South |CUST000002 |38.47          |1                 |Residential  

In [ ]:
enterprise_gold_features.write.mode("overwrite").parquet(str(GOLD_PATH / "enterprise_integrated_view"))
print("Gold layer saved successfully.")

Gold layer saved successfully.


In [ ]:
kpi_region = (
    enterprise_gold_features
    .groupBy("region")
    .agg(
        F.round(F.avg("daily_kwh_usage"), 2).alias("avg_daily_kwh_usage"),
        F.round(F.avg("daily_outage_minutes"), 2).alias("avg_daily_outage_minutes"),
        F.round(F.sum("daily_billed_amount"), 2).alias("total_billed_amount")
    )
    .orderBy("region")
)

kpi_region.show(truncate=False)

+-------+-------------------+------------------------+-------------------+
|region |avg_daily_kwh_usage|avg_daily_outage_minutes|total_billed_amount|
+-------+-------------------+------------------------+-------------------+
|Central|35.05              |0.43                    |36602.69           |
|East   |35.14              |0.37                    |32447.85           |
|North  |34.95              |0.41                    |40646.27           |
|South  |35.08              |0.38                    |36874.13           |
|West   |34.97              |0.35                    |36391.76           |
+-------+-------------------+------------------------+-------------------+



In [ ]:
kpi_region.toPandas().to_csv(OUTPUT_PATH / "regional_kpis.csv", index=False)
enterprise_gold_features.limit(1000).toPandas().to_csv(OUTPUT_PATH / "enterprise_gold_sample.csv", index=False)

print("Gold sample and KPI files exported.")

Gold sample and KPI files exported.


In [ ]:
ai_ready_df = (
    enterprise_gold_features
    .withColumn(
        "semantic_text",
        F.concat_ws(
            " ",
            F.lit("Customer"), F.col("customer_id"),
            F.lit("Region"), F.col("region"),
            F.lit("CustomerType"), F.col("customer_type"),
            F.lit("Date"), F.col("event_date").cast("string"),
            F.lit("Usage"), F.round(F.col("daily_kwh_usage"), 2).cast("string"),
            F.lit("OutageMinutes"), F.col("daily_outage_minutes").cast("string"),
            F.lit("BilledAmount"), F.round(F.col("daily_billed_amount"), 2).cast("string"),
            F.lit("Temperature"), F.round(F.col("temperature_f"), 2).cast("string"),
            F.lit("StormFlag"), F.col("storm_flag").cast("string")
        )
    )
    .withColumn("semantic_chunk_id", F.concat_ws("_", F.col("customer_id"), F.col("event_date").cast("string")))
)

ai_ready_df.select("semantic_chunk_id", "semantic_text").show(5, truncate=False)

+---------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+
|semantic_chunk_id    |semantic_text                                                                                                                                         |
+---------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+
|CUST002013_2023-05-21|Customer CUST002013 Region West CustomerType Commercial Date 2023-05-21 Usage 38.55 OutageMinutes 0 BilledAmount 0.0 Temperature 46.01 StormFlag 0    |
|CUST000048_2023-12-22|Customer CUST000048 Region East CustomerType Industrial Date 2023-12-22 Usage 34.11 OutageMinutes 0 BilledAmount 0.0 Temperature 55.98 StormFlag 0    |
|CUST002260_2023-04-19|Customer CUST002260 Region Central CustomerType Industrial Date 2023-04-19 Usage 26.84 OutageMinutes 0

In [ ]:
ai_ready_df.select(
    "semantic_chunk_id",
    "customer_id",
    "region",
    "customer_type",
    "event_date",
    "semantic_text"
).write.mode("overwrite").parquet(str(GOLD_PATH / "ai_ready_semantic_records"))

print("AI-ready semantic dataset saved successfully.")

AI-ready semantic dataset saved successfully.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

ml_df = enterprise_gold_features.select(
    "daily_kwh_usage",
    "daily_outage_minutes",
    "temperature_f",
    "rainfall_mm",
    "wind_speed_mph",
    "storm_flag",
    "usage_7day_avg",
    "usage_30day_avg",
    "is_high_outage_day"
).fillna(0).toPandas()

X = ml_df[
    [
        "daily_kwh_usage",
        "daily_outage_minutes",
        "temperature_f",
        "rainfall_mm",
        "wind_speed_mph",
        "storm_flag",
        "usage_7day_avg",
        "usage_30day_avg"
    ]
]
y = ml_df["is_high_outage_day"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19457
           1       1.00      1.00      1.00        16

    accuracy                           1.00     19473
   macro avg       1.00      1.00      1.00     19473
weighted avg       1.00      1.00      1.00     19473



In [ ]:
import json
from datetime import datetime

summary = {
    "bronze_customers": customers_bronze.count(),
    "bronze_meter_records": meter_bronze.count(),
    "silver_meter_records": meter_silver.count(),
    "gold_records": enterprise_gold_features.count(),
    "ai_ready_records": ai_ready_df.count(),
    "generated_at": datetime.now().isoformat()
}

with open(OUTPUT_PATH / "pipeline_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "bronze_customers": 5000,
  "bronze_meter_records": 100050,
  "silver_meter_records": 97361,
  "gold_records": 97361,
  "ai_ready_records": 97361,
  "generated_at": "2026-03-17T02:02:13.514351"
}


In [ ]:
spark.stop()
print("Pipeline completed successfully.")

Pipeline completed successfully.
